
## File 05 — Final Data Validation:


This notebook performs business-level validation across your E-Commerce Lakehouse.

Unlike File 04, which checks table availability, primary keys and basic relationships, this notebook checks whether your business calculations make sense.

We'll validate:

- Order totals against order-item totals.
- Successful and refunded payment amounts.
- Gold daily sales against Silver orders.
- Product revenue against order items.
- Customer lifetime value against payment records.
- Shipment delivery metrics.
- Inventory quantities and valuations.
- Final business reconciliation report.

In [0]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType
)
from pyspark.sql.window import Window

from datetime import datetime, timezone
import uuid


# ============================================================
# 2. CONFIGURATION
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

RUN_ID = str(uuid.uuid4())

TEST_TIMESTAMP = datetime.now(
    timezone.utc
).replace(tzinfo=None)

TOLERANCE = 0.02

VALIDATION_REPORT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "final_business_validation_report"
)

VALIDATION_SUMMARY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "final_business_validation_summary"
)

print("Configuration loaded successfully.")
print(f"Run ID: {RUN_ID}")


# ============================================================
# 3. SOURCE TABLES
# ============================================================

ORDERS = f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
ORDER_ITEMS = f"{CATALOG}.{SILVER_SCHEMA}.silver_order_items"
CUSTOMERS = f"{CATALOG}.{SILVER_SCHEMA}.silver_customers"
PRODUCTS = f"{CATALOG}.{SILVER_SCHEMA}.silver_products"
PAYMENTS = f"{CATALOG}.{SILVER_SCHEMA}.silver_payments"
SHIPMENTS = f"{CATALOG}.{SILVER_SCHEMA}.silver_shipments"
INVENTORY = f"{CATALOG}.{SILVER_SCHEMA}.silver_inventory"


# ============================================================
# 4. GOLD TABLES
# ============================================================

GOLD_DAILY_SALES = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_sales"
)

GOLD_PRODUCT_PERFORMANCE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_product_performance"
)

GOLD_CLV = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_customer_lifetime_value"
)

GOLD_PAYMENT_METRICS = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_payment_metrics"
)

GOLD_SHIPPING_METRICS = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_shipping_metrics"
)

GOLD_INVENTORY_STATUS = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_inventory_status"
)


# ============================================================
# 5. VALIDATION RESULT SCHEMA
# ============================================================

validation_schema = StructType([

    StructField("validation_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("validation_category", StringType(), True),
    StructField("validation_name", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("source_value", StringType(), True),
    StructField("target_value", StringType(), True),
    StructField("difference_value", DoubleType(), True),
    StructField("validation_status", StringType(), True),
    StructField("validation_message", StringType(), True),
    StructField("validated_at", TimestampType(), True)

])


# ============================================================
# 6. VALIDATION RESULT COLLECTION
# ============================================================

validation_results = []


def add_validation(
    category,
    name,
    source_table,
    target_table,
    source_value,
    target_value,
    difference,
    status,
    message
):

    validation_results.append({

        "validation_id": str(uuid.uuid4()),
        "run_id": RUN_ID,
        "validation_category": category,
        "validation_name": name,
        "source_table": source_table,
        "target_table": target_table,

        "source_value": (
            None if source_value is None
            else str(source_value)
        ),

        "target_value": (
            None if target_value is None
            else str(target_value)
        ),

        "difference_value": (
            None if difference is None
            else float(difference)
        ),

        "validation_status": status,
        "validation_message": message,
        "validated_at": TEST_TIMESTAMP

    })


# ============================================================
# 7. HELPER FUNCTIONS
# ============================================================

def require_table(table_name):

    if not spark.catalog.tableExists(table_name):

        raise RuntimeError(
            f"Required table is missing: {table_name}"
        )

    return spark.table(table_name)


def has_columns(df, required_columns):

    return all(
        column_name in df.columns
        for column_name in required_columns
    )


def first_available_column(df, candidates):

    for column_name in candidates:

        if column_name in df.columns:
            return column_name

    return None


def numeric_value(df, column_name):

    row = (
        df
        .agg(
            F.coalesce(
                F.sum(
                    F.col(column_name).cast("double")
                ),
                F.lit(0.0)
            ).alias("total")
        )
        .first()
    )

    return float(row["total"])


def safe_difference(source, target):

    return round(
        abs(float(source) - float(target)),
        2
    )


def record_not_tested(
    category,
    name,
    source_table,
    target_table,
    message
):

    add_validation(
        category,
        name,
        source_table,
        target_table,
        None,
        None,
        None,
        "NOT_TESTED",
        message
    )


def record_comparison(
    category,
    name,
    source_table,
    target_table,
    source_value,
    target_value,
    difference,
    passed,
    success_message,
    failure_message
):

    add_validation(
        category,
        name,
        source_table,
        target_table,
        source_value,
        target_value,
        difference,
        "PASSED" if passed else "FAILED",
        success_message if passed else failure_message
    )


# ============================================================
# 8. READ SILVER TABLES
# ============================================================

print("\nREADING SILVER TABLES")

orders_df = require_table(ORDERS)
items_df = require_table(ORDER_ITEMS)
customers_df = require_table(CUSTOMERS)
products_df = require_table(PRODUCTS)
payments_df = require_table(PAYMENTS)
shipments_df = require_table(SHIPMENTS)
inventory_df = require_table(INVENTORY)


# ============================================================
# 9. READ GOLD TABLES
# ============================================================

print("\nREADING GOLD TABLES")

gold_daily_df = require_table(GOLD_DAILY_SALES)
gold_product_df = require_table(GOLD_PRODUCT_PERFORMANCE)
gold_clv_df = require_table(GOLD_CLV)
gold_payment_df = require_table(GOLD_PAYMENT_METRICS)
gold_shipping_df = require_table(GOLD_SHIPPING_METRICS)
gold_inventory_df = require_table(GOLD_INVENTORY_STATUS)

print("All required Silver and Gold tables are available.")


# ============================================================
# 10. VALIDATE ORDER TOTALS AGAINST ORDER ITEMS
# ============================================================

print("\nVALIDATION 1: ORDER TOTALS")

required_order_columns = [
    "order_id",
    "order_total",
    "order_status"
]

required_item_columns = [
    "order_id",
    "line_total"
]

if (
    has_columns(orders_df, required_order_columns)
    and has_columns(items_df, required_item_columns)
):

    item_totals_df = (
        items_df
        .groupBy("order_id")
        .agg(
            F.sum(
                F.col("line_total").cast("double")
            ).alias("calculated_item_total")
        )
    )

    order_reconciliation_df = (
        orders_df
        .filter(
            F.upper(
                F.coalesce(
                    F.col("order_status"),
                    F.lit("")
                )
            ) != "CANCELLED"
        )
        .join(item_totals_df, "order_id", "left")
        .fillna({
            "calculated_item_total": 0.0
        })
        .withColumn(
            "difference",
            F.abs(
                F.col("order_total").cast("double")
                - F.col("calculated_item_total")
            )
        )
    )

    order_mismatch_count = (
        order_reconciliation_df
        .filter(F.col("difference") > TOLERANCE)
        .count()
    )

    order_count = order_reconciliation_df.count()

    record_comparison(
        "ORDERS",
        "Order total versus item total",
        ORDERS,
        ORDER_ITEMS,
        order_count - order_mismatch_count,
        order_count,
        order_mismatch_count,
        order_mismatch_count == 0,
        "All non-cancelled order totals match their item totals.",
        (
            "Order and item totals differ for some orders. "
            "Check tax, shipping, discounts and missing items."
        )
    )

else:

    record_not_tested(
        "ORDERS",
        "Order total versus item total",
        ORDERS,
        ORDER_ITEMS,
        "Required order_id, order_total, order_status "
        "or line_total columns are unavailable."
    )


# ============================================================
# 11. VALIDATE PAYMENT TOTALS
# ============================================================

print("\nVALIDATION 2: PAYMENT TOTALS")

required_payment_columns = [
    "order_id",
    "payment_status",
    "amount"
]

if (
    has_columns(orders_df, ["order_id", "order_total"])
    and has_columns(payments_df, required_payment_columns)
):

    payment_summary_df = (
        payments_df
        .groupBy("order_id")
        .agg(

            F.sum(
                F.when(
                    F.upper(F.col("payment_status")) == "SUCCESS",
                    F.col("amount").cast("double")
                ).otherwise(0.0)
            ).alias("successful_amount"),

            F.sum(
                F.when(
                    F.upper(F.col("payment_status")) == "REFUNDED",
                    F.col("amount").cast("double")
                ).otherwise(0.0)
            ).alias("refunded_amount"),

            F.sum(
                F.when(
                    F.upper(F.col("payment_status")) == "PENDING",
                    F.col("amount").cast("double")
                ).otherwise(0.0)
            ).alias("pending_amount")

        )
    )

    payment_reconciliation_df = (
        orders_df
        .join(payment_summary_df, "order_id", "left")
        .fillna({
            "successful_amount": 0.0,
            "refunded_amount": 0.0,
            "pending_amount": 0.0
        })
        .withColumn(
            "net_collected_amount",
            F.col("successful_amount")
            - F.col("refunded_amount")
        )
    )

    overpaid_orders = (
        payment_reconciliation_df
        .filter(
            F.col("net_collected_amount")
            > F.col("order_total").cast("double") + TOLERANCE
        )
        .count()
    )

    record_comparison(
        "PAYMENTS",
        "Net payment amount does not exceed order value",
        PAYMENTS,
        ORDERS,
        overpaid_orders,
        0,
        overpaid_orders,
        overpaid_orders == 0,
        "No overpaid orders detected.",
        (
            "Some orders have net payment amounts exceeding "
            "their order value. Review payment retries, "
            "refund handling and payment status conventions."
        )
    )

else:

    record_not_tested(
        "PAYMENTS",
        "Net payment amount does not exceed order value",
        PAYMENTS,
        ORDERS,
        "Required order or payment columns are unavailable."
    )


# ============================================================
# 12. VALIDATE GOLD DAILY SALES
# ============================================================

print("\nVALIDATION 3: GOLD DAILY SALES")

sales_date_source = first_available_column(
    orders_df,
    ["order_timestamp", "order_date"]
)

sales_date_gold = first_available_column(
    gold_daily_df,
    ["order_date", "sales_date", "date"]
)

sales_value_gold = first_available_column(
    gold_daily_df,
    [
        "gross_order_value",
        "gross_sales",
        "total_sales",
        "total_revenue",
        "sales_amount"
    ]
)

if (
    sales_date_source is not None
    and sales_date_gold is not None
    and sales_value_gold is not None
    and has_columns(orders_df, ["order_status", "order_total"])
):

    source_group_columns = ["order_date"]
    gold_group_columns = ["order_date"]

    silver_sales_base = (
        orders_df
        .filter(
            F.upper(
                F.coalesce(
                    F.col("order_status"),
                    F.lit("")
                )
            ) != "CANCELLED"
        )
        .withColumn(
            "order_date",
            F.to_date(F.col(sales_date_source))
        )
    )

    gold_sales_base = (
        gold_daily_df
        .withColumn(
            "order_date",
            F.to_date(F.col(sales_date_gold))
        )
    )

    if (
        "currency" in silver_sales_base.columns
        and "currency" in gold_sales_base.columns
    ):
        source_group_columns.append("currency")
        gold_group_columns.append("currency")

    silver_sales_df = (
        silver_sales_base
        .groupBy(*source_group_columns)
        .agg(
            F.sum(
                F.col("order_total").cast("double")
            ).alias("silver_sales")
        )
    )

    gold_sales_df = (
        gold_sales_base
        .groupBy(*gold_group_columns)
        .agg(
            F.sum(
                F.col(sales_value_gold).cast("double")
            ).alias("gold_sales")
        )
    )

    sales_reconciliation_df = (
        silver_sales_df.alias("s")
        .join(
            gold_sales_df.alias("g"),
            source_group_columns,
            "full"
        )
        .fillna({
            "silver_sales": 0.0,
            "gold_sales": 0.0
        })
        .withColumn(
            "difference",
            F.abs(
                F.col("silver_sales")
                - F.col("gold_sales")
            )
        )
    )

    sales_mismatch_count = (
        sales_reconciliation_df
        .filter(F.col("difference") > TOLERANCE)
        .count()
    )

    silver_sales_total = numeric_value(
        silver_sales_df,
        "silver_sales"
    )

    gold_sales_total = numeric_value(
        gold_sales_df,
        "gold_sales"
    )

    record_comparison(
        "GOLD_SALES",
        "Silver orders versus Gold daily sales",
        ORDERS,
        GOLD_DAILY_SALES,
        silver_sales_total,
        gold_sales_total,
        safe_difference(
            silver_sales_total,
            gold_sales_total
        ),
        sales_mismatch_count == 0,
        "Gold daily sales reconcile with Silver order totals.",
        (
            f"Found {sales_mismatch_count} mismatched "
            "date/currency groups between Silver and Gold sales."
        )
    )

else:

    record_not_tested(
        "GOLD_SALES",
        "Silver orders versus Gold daily sales",
        ORDERS,
        GOLD_DAILY_SALES,
        (
            "Required sales date or Gold sales amount column "
            "is unavailable. Inspect both table schemas."
        )
    )


# ============================================================
# 13. VALIDATE GOLD PRODUCT REVENUE
# ============================================================

print("\nVALIDATION 4: PRODUCT REVENUE")

product_revenue_column = first_available_column(
    gold_product_df,
    [
        "total_revenue",
        "product_revenue",
        "gross_revenue",
        "revenue",
        "total_sales"
    ]
)

if (
    product_revenue_column is not None
    and has_columns(
        items_df,
        ["order_id", "product_id", "line_total"]
    )
    and has_columns(
        orders_df,
        ["order_id", "order_status"]
    )
    and "product_id" in gold_product_df.columns
):

    valid_order_ids = (
        orders_df
        .filter(
            F.upper(
                F.coalesce(
                    F.col("order_status"),
                    F.lit("")
                )
            ) != "CANCELLED"
        )
        .select("order_id")
        .distinct()
    )

    silver_product_revenue_df = (
        items_df
        .join(valid_order_ids, "order_id", "inner")
        .groupBy("product_id")
        .agg(
            F.sum(
                F.col("line_total").cast("double")
            ).alias("silver_revenue")
        )
    )

    gold_product_revenue_df = (
        gold_product_df
        .groupBy("product_id")
        .agg(
            F.sum(
                F.col(product_revenue_column).cast("double")
            ).alias("gold_revenue")
        )
    )

    product_reconciliation_df = (
        silver_product_revenue_df.alias("s")
        .join(
            gold_product_revenue_df.alias("g"),
            "product_id",
            "full"
        )
        .fillna({
            "silver_revenue": 0.0,
            "gold_revenue": 0.0
        })
        .withColumn(
            "difference",
            F.abs(
                F.col("silver_revenue")
                - F.col("gold_revenue")
            )
        )
    )

    product_mismatch_count = (
        product_reconciliation_df
        .filter(F.col("difference") > TOLERANCE)
        .count()
    )

    silver_product_total = numeric_value(
        silver_product_revenue_df,
        "silver_revenue"
    )

    gold_product_total = numeric_value(
        gold_product_revenue_df,
        "gold_revenue"
    )

    record_comparison(
        "GOLD_PRODUCTS",
        "Product revenue reconciliation",
        ORDER_ITEMS,
        GOLD_PRODUCT_PERFORMANCE,
        silver_product_total,
        gold_product_total,
        safe_difference(
            silver_product_total,
            gold_product_total
        ),
        product_mismatch_count == 0,
        "Product revenue matches Silver order item revenue.",
        (
            f"Found {product_mismatch_count} products "
            "with different Silver and Gold revenue."
        )
    )

else:

    record_not_tested(
        "GOLD_PRODUCTS",
        "Product revenue reconciliation",
        ORDER_ITEMS,
        GOLD_PRODUCT_PERFORMANCE,
        (
            "Required product_id or supported Gold revenue "
            "column is unavailable."
        )
    )


# ============================================================
# 14. VALIDATE GOLD CUSTOMER LIFETIME VALUE
# ============================================================

print("\nVALIDATION 5: CUSTOMER LIFETIME VALUE")

required_clv_gold_columns = [
    "customer_id",
    "total_orders",
    "total_order_value",
    "historical_customer_lifetime_value"
]

required_clv_order_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "order_total"
]

if (
    has_columns(gold_clv_df, required_clv_gold_columns)
    and has_columns(orders_df, required_clv_order_columns)
    and "customer_id" in customers_df.columns
):

    # Gold customer metrics exclude cancelled orders.
    # This matches the known Gold total_order_value:
    # 968,162,183.01 across the supplied dataset.

    eligible_orders_df = (
        orders_df
        .filter(
            F.upper(
                F.coalesce(
                    F.col("order_status"),
                    F.lit("")
                )
            ) != "CANCELLED"
        )
    )

    # Aggregate eligible Silver orders per customer.

    source_customer_orders_df = (
        eligible_orders_df
        .groupBy("customer_id")
        .agg(
            F.countDistinct("order_id").alias(
                "source_total_orders"
            ),
            F.sum(
                F.col("order_total").cast("double")
            ).alias("source_total_order_value")
        )
    )

    # Include customers with no eligible orders.

    source_customer_metrics_df = (
        customers_df
        .select("customer_id")
        .distinct()
        .join(
            source_customer_orders_df,
            "customer_id",
            "left"
        )
        .fillna({
            "source_total_orders": 0,
            "source_total_order_value": 0.0
        })
    )

    # Aggregate Gold by customer_id.
    # Count rows to detect unexpected duplicate customer records.

    gold_customer_metrics_df = (
        gold_clv_df
        .groupBy("customer_id")
        .agg(
            F.sum(
                F.col("total_orders").cast("long")
            ).alias("gold_total_orders"),
            F.sum(
                F.col("total_order_value").cast("double")
            ).alias("gold_total_order_value"),
            F.count("*").alias("gold_customer_rows"),
            F.max(
                F.col(
                    "historical_customer_lifetime_value"
                ).cast("double")
            ).alias("historical_clv")
        )
    )

    # Compare every customer, including missing IDs.

    customer_comparison_df = (
        source_customer_metrics_df.alias("s")
        .join(
            gold_customer_metrics_df.alias("g"),
            "customer_id",
            "full"
        )
        .fillna({
            "source_total_orders": 0,
            "source_total_order_value": 0.0,
            "gold_total_orders": 0,
            "gold_total_order_value": 0.0,
            "gold_customer_rows": 0
        })
        .withColumn(
            "order_count_difference",
            F.abs(
                F.col("source_total_orders")
                - F.col("gold_total_orders")
            )
        )
        .withColumn(
            "order_value_difference",
            F.abs(
                F.col("source_total_order_value")
                - F.col("gold_total_order_value")
            )
        )
    )

    customer_mismatch_count = (
        customer_comparison_df
        .filter(
            (F.col("order_count_difference") != 0)
            |
            (F.col("order_value_difference") > TOLERANCE)
            |
            (F.col("gold_customer_rows") == 0)
            |
            (F.col("gold_customer_rows") > 1)
        )
        .count()
    )

    source_customer_count = (
        source_customer_metrics_df
        .count()
    )

    gold_customer_count = (
        gold_clv_df
        .select("customer_id")
        .distinct()
        .count()
    )

    source_order_value_total = numeric_value(
        source_customer_metrics_df,
        "source_total_order_value"
    )

    gold_order_value_total = numeric_value(
        gold_clv_df,
        "total_order_value"
    )

    # Historical CLV is not assumed to equal order value
    # or net collected payment.
    # Check that the stored metric is non-negative and populated.

    invalid_historical_clv_count = (
        gold_clv_df
        .filter(
            F.col("historical_customer_lifetime_value").isNull()
            |
            (
                F.col(
                    "historical_customer_lifetime_value"
                ) < 0
            )
        )
        .count()
    )

    customer_validation_passed = (
        customer_mismatch_count == 0
        and invalid_historical_clv_count == 0
        and source_customer_count == gold_customer_count
    )

    record_comparison(
        "GOLD_CUSTOMERS",
        "Customer lifetime value reconciliation",
        ORDERS,
        GOLD_CLV,
        source_order_value_total,
        gold_order_value_total,
        safe_difference(
            source_order_value_total,
            gold_order_value_total
        ),
        customer_validation_passed,
        (
            "Customer-level Gold order counts and order values "
            "match eligible Silver orders. Historical CLV values "
            "are present and non-negative."
        ),
        (
            f"Customer mismatches: {customer_mismatch_count}; "
            f"invalid historical CLV records: "
            f"{invalid_historical_clv_count}; "
            f"source customers: {source_customer_count}; "
            f"Gold customers: {gold_customer_count}. "
            "Review customer-level reconciliation."
        )
    )

    print("Source customers:", source_customer_count)
    print("Gold customers:", gold_customer_count)
    print("Customer mismatches:", customer_mismatch_count)
    print(
        "Invalid historical CLV:",
        invalid_historical_clv_count
    )

else:

    record_not_tested(
        "GOLD_CUSTOMERS",
        "Customer lifetime value reconciliation",
        ORDERS,
        GOLD_CLV,
        (
            "Required customer_id, order_id, order_status, "
            "order_total, total_orders, total_order_value or "
            "historical_customer_lifetime_value columns "
            "are unavailable."
        )
    )


# ============================================================
# 15. VALIDATE GOLD SHIPPING METRICS
# ============================================================

print("\nVALIDATION 6: SHIPPING METRICS")

required_silver_shipping_columns = [
    "shipment_status",
    "shipped_date"
]

required_gold_shipping_columns = [
    "shipment_date",
    "total_shipments",
    "delivered_shipments",
    "shipped_shipments",
    "in_transit_shipments",
    "out_for_delivery_shipments",
    "processing_shipments",
    "pending_shipments",
    "cancelled_shipments",
    "returned_shipments"
]

if (
    has_columns(
        shipments_df,
        required_silver_shipping_columns
    )
    and has_columns(
        gold_shipping_df,
        required_gold_shipping_columns
    )
):

    # Convert source shipped_date into a date.
    source_shipping_base_df = (
        shipments_df
        .withColumn(
            "shipment_date",
            F.to_date("shipped_date")
        )
    )

    invalid_shipment_date_count = (
        source_shipping_base_df
        .filter(F.col("shipment_date").isNull())
        .count()
    )

    # Gold daily metrics cannot be compared against a
    # source record with no valid shipment date.
    valid_source_shipping_df = (
        source_shipping_base_df
        .filter(F.col("shipment_date").isNotNull())
    )

    # Aggregate Silver records by shipment date.

    source_shipping_summary_df = (
        valid_source_shipping_df
        .groupBy("shipment_date")
        .agg(

            F.count("*").alias(
                "source_total_shipments"
            ),

            F.sum(
                F.when(
                    F.upper(F.col("shipment_status")) == "DELIVERED",
                    1
                ).otherwise(0)
            ).alias("source_delivered_shipments"),

            F.sum(
                F.when(
                    F.upper(F.col("shipment_status")) == "SHIPPED",
                    1
                ).otherwise(0)
            ).alias("source_shipped_shipments"),

            F.sum(
                F.when(
                    F.upper(F.col("shipment_status")) == "IN_TRANSIT",
                    1
                ).otherwise(0)
            ).alias("source_in_transit_shipments"),

            F.sum(
                F.when(
                    F.upper(F.col("shipment_status")) == "OUT_FOR_DELIVERY",
                    1
                ).otherwise(0)
            ).alias("source_out_for_delivery_shipments"),

            F.sum(
                F.when(
                    F.upper(F.col("shipment_status")) == "PROCESSING",
                    1
                ).otherwise(0)
            ).alias("source_processing_shipments"),

            F.sum(
                F.when(
                    F.upper(F.col("shipment_status")) == "PENDING",
                    1
                ).otherwise(0)
            ).alias("source_pending_shipments"),

            F.sum(
                F.when(
                    F.upper(F.col("shipment_status")) == "CANCELLED",
                    1
                ).otherwise(0)
            ).alias("source_cancelled_shipments"),

            # Project mapping:
            # RETURN_TO_ORIGIN -> returned_shipments

            F.sum(
                F.when(
                    F.upper(F.col("shipment_status"))
                    .isin("RETURNED", "RETURN_TO_ORIGIN"),
                    1
                ).otherwise(0)
            ).alias("source_returned_shipments"),

            F.sum(
                F.when(
                    F.upper(F.col("shipment_status"))
                    == "UNKNOWN_STATUS",
                    1
                ).otherwise(0)
            ).alias("source_unknown_shipments")

        )
    )

    # Aggregate Gold by date.
    # Grouping prevents accidental duplicate-date rows
    # from being compared as individual records.

    gold_shipping_summary_df = (
        gold_shipping_df
        .groupBy("shipment_date")
        .agg(

            F.sum("total_shipments").alias(
                "gold_total_shipments"
            ),

            F.sum("delivered_shipments").alias(
                "gold_delivered_shipments"
            ),

            F.sum("shipped_shipments").alias(
                "gold_shipped_shipments"
            ),

            F.sum("in_transit_shipments").alias(
                "gold_in_transit_shipments"
            ),

            F.sum("out_for_delivery_shipments").alias(
                "gold_out_for_delivery_shipments"
            ),

            F.sum("processing_shipments").alias(
                "gold_processing_shipments"
            ),

            F.sum("pending_shipments").alias(
                "gold_pending_shipments"
            ),

            F.sum("cancelled_shipments").alias(
                "gold_cancelled_shipments"
            ),

            F.sum("returned_shipments").alias(
                "gold_returned_shipments"
            )

        )
    )

    # Compare Silver and Gold by shipment date.

    shipping_comparison_df = (
        source_shipping_summary_df.alias("s")
        .join(
            gold_shipping_summary_df.alias("g"),
            "shipment_date",
            "full"
        )
        .fillna(0)
    )

    # Compare total shipments and every supported status metric.

    shipping_comparison_df = (
        shipping_comparison_df
        .withColumn(
            "total_difference",
            F.abs(
                F.col("source_total_shipments")
                - F.col("gold_total_shipments")
            )
        )
        .withColumn(
            "delivered_difference",
            F.abs(
                F.col("source_delivered_shipments")
                - F.col("gold_delivered_shipments")
            )
        )
        .withColumn(
            "shipped_difference",
            F.abs(
                F.col("source_shipped_shipments")
                - F.col("gold_shipped_shipments")
            )
        )
        .withColumn(
            "in_transit_difference",
            F.abs(
                F.col("source_in_transit_shipments")
                - F.col("gold_in_transit_shipments")
            )
        )
        .withColumn(
            "out_for_delivery_difference",
            F.abs(
                F.col("source_out_for_delivery_shipments")
                - F.col("gold_out_for_delivery_shipments")
            )
        )
        .withColumn(
            "processing_difference",
            F.abs(
                F.col("source_processing_shipments")
                - F.col("gold_processing_shipments")
            )
        )
        .withColumn(
            "pending_difference",
            F.abs(
                F.col("source_pending_shipments")
                - F.col("gold_pending_shipments")
            )
        )
        .withColumn(
            "cancelled_difference",
            F.abs(
                F.col("source_cancelled_shipments")
                - F.col("gold_cancelled_shipments")
            )
        )
        .withColumn(
            "returned_difference",
            F.abs(
                F.col("source_returned_shipments")
                - F.col("gold_returned_shipments")
            )
        )
    )

    difference_columns = [
        "total_difference",
        "delivered_difference",
        "shipped_difference",
        "in_transit_difference",
        "out_for_delivery_difference",
        "processing_difference",
        "pending_difference",
        "cancelled_difference",
        "returned_difference"
    ]

    mismatch_condition = None

    for column_name in difference_columns:

        condition = F.col(column_name) != 0

        mismatch_condition = (
            condition
            if mismatch_condition is None
            else mismatch_condition | condition
        )

    shipping_mismatch_count = (
        shipping_comparison_df
        .filter(mismatch_condition)
        .count()
    )

    source_shipping_total = numeric_value(
        source_shipping_summary_df,
        "source_total_shipments"
    )

    gold_shipping_total = numeric_value(
        gold_shipping_summary_df,
        "gold_total_shipments"
    )

    shipping_validation_passed = (
        shipping_mismatch_count == 0
        and invalid_shipment_date_count == 0
    )

    record_comparison(
        "GOLD_SHIPPING",
        "Shipment status count reconciliation",
        SHIPMENTS,
        GOLD_SHIPPING_METRICS,
        source_shipping_total,
        gold_shipping_total,
        safe_difference(
            source_shipping_total,
            gold_shipping_total
        ),
        shipping_validation_passed,
        (
            "Daily Silver shipment totals and supported status "
            "counts match Gold shipping metrics."
        ),
        (
            f"Daily mismatches: {shipping_mismatch_count}; "
            f"invalid Silver shipment dates: "
            f"{invalid_shipment_date_count}. "
            "Check date aggregation and shipment status mappings."
        )
    )

    print("Source shipment total:", source_shipping_total)
    print("Gold shipment total:", gold_shipping_total)
    print("Daily mismatches:", shipping_mismatch_count)
    print(
        "Invalid shipment dates:",
        invalid_shipment_date_count
    )

else:

    record_not_tested(
        "GOLD_SHIPPING",
        "Shipment status count reconciliation",
        SHIPMENTS,
        GOLD_SHIPPING_METRICS,
        (
            "Required Silver shipped_date/status or Gold "
            "shipment_date and daily shipment metric columns "
            "are unavailable."
        )
    )


# ============================================================
# 16. VALIDATE NEGATIVE INVENTORY QUANTITIES
# ============================================================

print("\nVALIDATION 7: NEGATIVE INVENTORY")

if "quantity_on_hand" in inventory_df.columns:

    negative_inventory_count = (
        inventory_df
        .filter(
            F.col("quantity_on_hand").cast("double") < 0
        )
        .count()
    )

    record_comparison(
        "INVENTORY",
        "Negative inventory quantity",
        INVENTORY,
        GOLD_INVENTORY_STATUS,
        negative_inventory_count,
        0,
        negative_inventory_count,
        negative_inventory_count == 0,
        "No negative inventory quantities found.",
        (
            f"Found {negative_inventory_count} inventory "
            "records with negative quantities."
        )
    )

else:

    record_not_tested(
        "INVENTORY",
        "Negative inventory quantity",
        INVENTORY,
        GOLD_INVENTORY_STATUS,
        "Silver inventory does not contain quantity_on_hand."
    )


# ============================================================
# 17. VALIDATE GOLD INVENTORY QUANTITIES
# ============================================================

print("\nVALIDATION 8: GOLD INVENTORY QUANTITIES")

gold_inventory_quantity_column = first_available_column(
    gold_inventory_df,
    [
        "total_quantity_on_hand",
        "total_inventory_quantity",
        "quantity_on_hand"
    ]
)

if (
    has_columns(
        inventory_df,
        [
            "product_id",
            "warehouse_id",
            "snapshot_date",
            "quantity_on_hand"
        ]
    )
    and "product_id" in gold_inventory_df.columns
    and gold_inventory_quantity_column is not None
):

    inventory_window = (
        Window
        .partitionBy("product_id", "warehouse_id")
        .orderBy(F.col("snapshot_date").desc())
    )

    latest_inventory_df = (
        inventory_df
        .withColumn(
            "_row_number",
            F.row_number().over(inventory_window)
        )
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

    silver_inventory_summary = (
        latest_inventory_df
        .groupBy("product_id")
        .agg(
            F.sum(
                F.col("quantity_on_hand").cast("double")
            ).alias("silver_quantity")
        )
    )

    gold_inventory_summary = (
        gold_inventory_df
        .groupBy("product_id")
        .agg(
            F.sum(
                F.col(
                    gold_inventory_quantity_column
                ).cast("double")
            ).alias("gold_quantity")
        )
    )

    inventory_comparison = (
        silver_inventory_summary.alias("s")
        .join(
            gold_inventory_summary.alias("g"),
            "product_id",
            "full"
        )
        .fillna({
            "silver_quantity": 0.0,
            "gold_quantity": 0.0
        })
        .withColumn(
            "difference",
            F.abs(
                F.col("silver_quantity")
                - F.col("gold_quantity")
            )
        )
    )

    inventory_mismatch_count = (
        inventory_comparison
        .filter(F.col("difference") > TOLERANCE)
        .count()
    )

    silver_inventory_total = numeric_value(
        silver_inventory_summary,
        "silver_quantity"
    )

    gold_inventory_total = numeric_value(
        gold_inventory_summary,
        "gold_quantity"
    )

    record_comparison(
        "GOLD_INVENTORY",
        "Latest Silver inventory versus Gold inventory quantity",
        INVENTORY,
        GOLD_INVENTORY_STATUS,
        silver_inventory_total,
        gold_inventory_total,
        safe_difference(
            silver_inventory_total,
            gold_inventory_total
        ),
        inventory_mismatch_count == 0,
        (
            "Latest Silver inventory quantities match "
            "Gold product inventory quantities."
        ),
        (
            f"Found {inventory_mismatch_count} products "
            "with different Silver and Gold inventory quantities."
        )
    )

else:

    record_not_tested(
        "GOLD_INVENTORY",
        "Latest Silver inventory versus Gold inventory quantity",
        INVENTORY,
        GOLD_INVENTORY_STATUS,
        (
            "Required inventory columns or Gold inventory "
            "quantity column are unavailable."
        )
    )


# ============================================================
# 18. VALIDATE INVENTORY VALUATION
# ============================================================

print("\nVALIDATION 9: INVENTORY VALUATION")

gold_inventory_value_column = first_available_column(
    gold_inventory_df,
    [
        "inventory_cost_value",
        "total_inventory_value",
        "total_cost_value"
    ]
)

product_cost_column = first_available_column(
    products_df,
    [
        "cost_price",
        "unit_cost",
        "product_cost",
        "cost"
    ]
)

if (
    gold_inventory_value_column is not None
    and product_cost_column is not None
    and has_columns(
        inventory_df,
        [
            "product_id",
            "warehouse_id",
            "snapshot_date",
            "quantity_on_hand"
        ]
    )
    and "product_id" in products_df.columns
    and "product_id" in gold_inventory_df.columns
):

    latest_inventory_for_value = (
        inventory_df
        .withColumn(
            "_row_number",
            F.row_number().over(
                Window
                .partitionBy("product_id", "warehouse_id")
                .orderBy(F.col("snapshot_date").desc())
            )
        )
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

    # Check whether product cost is unique before joining.
    duplicate_product_cost_count = (
        products_df
        .groupBy("product_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if duplicate_product_cost_count > 0:

        record_not_tested(
            "GOLD_INVENTORY",
            "Inventory cost valuation reconciliation",
            INVENTORY,
            GOLD_INVENTORY_STATUS,
            (
                f"Found {duplicate_product_cost_count} duplicate "
                "product IDs in Silver products. Cost valuation "
                "was not tested because product cost selection "
                "would be ambiguous."
            )
        )

    else:

        product_costs_df = (
            products_df
            .select(
                "product_id",
                F.col(product_cost_column)
                .cast("double")
                .alias("unit_cost")
            )
        )

        silver_inventory_value_df = (
            latest_inventory_for_value
            .join(product_costs_df, "product_id", "inner")
            .withColumn(
                "inventory_value",
                F.col("quantity_on_hand").cast("double")
                * F.col("unit_cost")
            )
            .groupBy("product_id")
            .agg(
                F.sum("inventory_value").alias(
                    "silver_inventory_value"
                )
            )
        )

        gold_inventory_value_df = (
            gold_inventory_df
            .groupBy("product_id")
            .agg(
                F.sum(
                    F.col(
                        gold_inventory_value_column
                    ).cast("double")
                ).alias("gold_inventory_value")
            )
        )

        valuation_comparison = (
            silver_inventory_value_df.alias("s")
            .join(
                gold_inventory_value_df.alias("g"),
                "product_id",
                "full"
            )
            .fillna({
                "silver_inventory_value": 0.0,
                "gold_inventory_value": 0.0
            })
            .withColumn(
                "difference",
                F.abs(
                    F.col("silver_inventory_value")
                    - F.col("gold_inventory_value")
                )
            )
        )

        valuation_mismatch_count = (
            valuation_comparison
            .filter(F.col("difference") > TOLERANCE)
            .count()
        )

        silver_value_total = numeric_value(
            silver_inventory_value_df,
            "silver_inventory_value"
        )

        gold_value_total = numeric_value(
            gold_inventory_value_df,
            "gold_inventory_value"
        )

        record_comparison(
            "GOLD_INVENTORY",
            "Inventory cost valuation reconciliation",
            INVENTORY,
            GOLD_INVENTORY_STATUS,
            silver_value_total,
            gold_value_total,
            safe_difference(
                silver_value_total,
                gold_value_total
            ),
            valuation_mismatch_count == 0,
            "Silver inventory cost valuation matches Gold.",
            (
                f"Found {valuation_mismatch_count} products "
                "with different inventory cost valuations."
            )
        )

else:

    record_not_tested(
        "GOLD_INVENTORY",
        "Inventory cost valuation reconciliation",
        INVENTORY,
        GOLD_INVENTORY_STATUS,
        (
            "Required product cost or Gold inventory value "
            "columns are unavailable. No valuation comparison "
            "was performed."
        )
    )


# ============================================================
# 19. BUILD VALIDATION REPORT
# ============================================================

print("\nBUILDING VALIDATION REPORT")

validation_report_df = spark.createDataFrame(
    validation_results,
    schema=validation_schema
)


# ============================================================
# 20. SAVE VALIDATION REPORT
# ============================================================

print("\nSAVING VALIDATION REPORT")

(
    validation_report_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(VALIDATION_REPORT_TABLE)
)


# ============================================================
# 21. BUILD VALIDATION SUMMARY
# ============================================================

print("\nBUILDING VALIDATION SUMMARY")

validation_summary_df = (
    validation_report_df
    .groupBy("run_id")
    .agg(

        F.count("*").alias("total_validations"),

        F.sum(
            F.when(
                F.col("validation_status") == "PASSED",
                1
            ).otherwise(0)
        ).alias("passed_validations"),

        F.sum(
            F.when(
                F.col("validation_status") == "FAILED",
                1
            ).otherwise(0)
        ).alias("failed_validations"),

        F.sum(
            F.when(
                F.col("validation_status") == "NOT_TESTED",
                1
            ).otherwise(0)
        ).alias("not_tested_validations")

    )
    .withColumn(
        "validation_pass_rate",
        F.when(
            (
                F.col("passed_validations")
                + F.col("failed_validations")
            ) > 0,
            F.round(
                F.col("passed_validations")
                / (
                    F.col("passed_validations")
                    + F.col("failed_validations")
                ) * 100,
                2
            )
        ).otherwise(0.0)
    )
    .withColumn(
        "overall_status",
        F.when(
            F.col("failed_validations") > 0,
            "FAILED"
        )
        .when(
            F.col("not_tested_validations") > 0,
            "INCOMPLETE"
        )
        .otherwise("PASSED")
    )
    .withColumn(
        "validated_at",
        F.lit(TEST_TIMESTAMP).cast("timestamp")
    )
)


# ============================================================
# 22. SAVE VALIDATION SUMMARY
# ============================================================

print("\nSAVING VALIDATION SUMMARY")

(
    validation_summary_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(VALIDATION_SUMMARY_TABLE)
)


# ============================================================
# 23. DISPLAY VALIDATION RESULTS
# ============================================================

print("\nFINAL BUSINESS VALIDATION REPORT")

display(
    validation_report_df
    .orderBy(
        "validation_category",
        "validation_name"
    )
)


# ============================================================
# 24. DISPLAY FAILED VALIDATIONS
# ============================================================

print("\nFAILED BUSINESS VALIDATIONS")

display(
    validation_report_df
    .filter(
        F.col("validation_status") == "FAILED"
    )
)


# ============================================================
# 25. DISPLAY NOT TESTED VALIDATIONS
# ============================================================

print("\nVALIDATIONS NOT EXECUTED")

display(
    validation_report_df
    .filter(
        F.col("validation_status") == "NOT_TESTED"
    )
)


# ============================================================
# 26. DISPLAY FINAL SUMMARY
# ============================================================

summary = validation_summary_df.first()

print("\n" + "=" * 70)
print("FINAL BUSINESS VALIDATION SUMMARY")
print("=" * 70)

print(f"Run ID: {RUN_ID}")

print(
    f"Total validations: "
    f"{summary['total_validations']}"
)

print(
    f"Passed validations: "
    f"{summary['passed_validations']}"
)

print(
    f"Failed validations: "
    f"{summary['failed_validations']}"
)

print(
    f"Not tested validations: "
    f"{summary['not_tested_validations']}"
)

print(
    f"Pass rate: "
    f"{summary['validation_pass_rate']}%"
)

print(
    f"Overall status: "
    f"{summary['overall_status']}"
)

print("=" * 70)


# ============================================================
# 27. FINAL STATUS
# ============================================================

print(
    f"Validation report table: "
    f"{VALIDATION_REPORT_TABLE}"
)

print(
    f"Validation summary table: "
    f"{VALIDATION_SUMMARY_TABLE}"
)

if summary["overall_status"] == "PASSED":

    print("FINAL BUSINESS VALIDATION PASSED.")

elif summary["overall_status"] == "FAILED":

    print(
        "FINAL BUSINESS VALIDATION FAILED. "
        "Review failed validation records."
    )

else:

    print(
        "FINAL BUSINESS VALIDATION INCOMPLETE. "
        "Review NOT_TESTED validation records."
    )

print("\nFILE 05 EXECUTION COMPLETED.")



### Identify the two untested validations

In [0]:
%sql
SELECT
    validation_category,
    validation_name,
    source_table,
    target_table,
    validation_message
FROM workspace.ecommerce_gold.final_business_validation_report
WHERE run_id = '996dc10c-5a90-433c-be0f-78ff39c91b6a'
  AND validation_status = 'NOT_TESTED'
ORDER BY validation_category;


### Inspect the Customer Lifetime Value table

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_customer_lifetime_value;

SELECT *
FROM workspace.ecommerce_gold.gold_customer_lifetime_value
LIMIT 5;


### inspect the source payments table

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.silver_payments;

SELECT *
FROM workspace.ecommerce_silver.silver_payments
LIMIT 5;


### Inspect the Shipping Metrics table

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_shipping_metrics;

SELECT *
FROM workspace.ecommerce_gold.gold_shipping_metrics
LIMIT 10;


### And inspect the source:

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.silver_shipments;

SELECT *
FROM workspace.ecommerce_silver.silver_shipments
LIMIT 5;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_customer_lifetime_value;

SELECT *
FROM workspace.ecommerce_gold.gold_customer_lifetime_value
LIMIT 10;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.silver_orders;


### Customer CLV — inspect order statuses


In [0]:
%sql

SELECT
    order_status,
    COUNT(*) AS order_count,
    ROUND(SUM(order_total), 2) AS total_order_value
FROM workspace.ecommerce_silver.silver_orders
GROUP BY order_status
ORDER BY order_status;

In [0]:
%sql

SELECT
    COUNT(*) AS gold_customers,
    COUNT(DISTINCT customer_id) AS unique_customers,
    ROUND(SUM(total_order_value), 2) AS gold_total_order_value,
    ROUND(SUM(historical_customer_lifetime_value), 2) AS historical_clv
FROM workspace.ecommerce_gold.gold_customer_lifetime_value;


### Shipping — identify actual status mappings

In [0]:
%sql

SELECT
    shipment_status,
    COUNT(*) AS shipment_count
FROM workspace.ecommerce_silver.silver_shipments
GROUP BY shipment_status
ORDER BY shipment_status;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_shipping_metrics;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_customer_lifetime_value;

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_gold.gold_customer_lifetime_value
LIMIT 10;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_silver.silver_orders;

In [0]:
%sql
SELECT
    shipment_status,
    COUNT(*) AS shipment_count
FROM workspace.ecommerce_silver.silver_shipments
GROUP BY shipment_status
ORDER BY shipment_status;

## SQL — Final validation results


### Overall validation

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_gold.final_business_validation_summary
ORDER BY validated_at DESC;


### Failed business validations

In [0]:
%sql
SELECT
    validation_category,
    validation_name,
    source_value,
    target_value,
    difference_value,
    validation_message
FROM workspace.ecommerce_gold.final_business_validation_report
WHERE validation_status = 'FAILED'
ORDER BY validated_at DESC;


### Validation category summary

In [0]:
%sql
SELECT
    validation_category,
    COUNT(*) AS total_checks,
    SUM(
        CASE
            WHEN validation_status = 'PASSED'
            THEN 1 ELSE 0
        END
    ) AS passed_checks,
    SUM(
        CASE
            WHEN validation_status = 'FAILED'
            THEN 1 ELSE 0
        END
    ) AS failed_checks
FROM workspace.ecommerce_gold.final_business_validation_report
GROUP BY validation_category
ORDER BY validation_category;